# Strategy card: an edge or a fluke?

Run this before a strategy goes into a paper portfolio. One call takes a sleeve (or any return series), runs it as the paper runtime would (alone, full size, with its costs and funding), and asks the same questions of every strategy (`src/research/strategy_checks.py`; pass marks in `research/strategy_standard.toml`, fixed in advance):

- **Is the mean return real?** Newey-West t-statistic, Sharpe ratio with a block-bootstrap interval.
- **Is it more than the best of many tries?** Deflated Sharpe (every trial in the ledger for its family counts), Romano-Wolf p-values across the parameter grid, probability of backtest overfitting (CSCV).
- **Is it stable?** Both halves, the share of positive years, the Sharpe without its best year, a rolling one-year Sharpe.
- **Does the timing earn it?** The same positions shifted months in time. If they do as well, the return is the market's drift.
- **Is it better than buy-and-hold?** A paired bootstrap of the Sharpe difference.
- **How good is it, really?** A Bayesian shrunk Sharpe (prior: no edge, sd 0.5 a year). Reported, not a gate: it is the number to expect live.

Where it sits: idea and pre-registration → signal card (`signal_card.ipynb`) → strategy test with costs → **strategy card** → `python main.py --portfolio-check <config> --require-strategy-cards` → paper book.

| Verdict | Meaning | What to do |
| --- | --- | --- |
| evidence of an edge | every check that applies passes | can go to a paper book |
| weak | the mean is significant, something else fails | paper only as a written-down exception (say which check fails and why it is accepted) |
| no evidence | the mean is not distinguishable from zero | does not go to a paper book |
| short history | under 3 years | collect more history first |

`RECORD = False` keeps experiments out of the trial ledger. Set it to `True` for the card you want the gate (`--portfolio-check`) to see.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import os; os.chdir(ROOT)                      # configs, caches and the ledger are read relative to the repo
warnings.filterwarnings("ignore")
import pandas as pd
from src.portfolio.sleeves import SleeveSpec
from src.research import strategy_checks as stc

RECORD = False                                 # True: note the card in the trial ledger, where --portfolio-check finds it
stc.load_standard()

## 1. A candidate sleeve (the usual case)

Write the sleeve exactly as it would go into a portfolio config. The config only supplies the instrument's costs; the sleeve doesn't have to be in it.
`variants="grid"` also runs every point of the strategy's catalog grid: those are the configurations a search would have tried, and they feed the deflated Sharpe, Romano-Wolf and the overfitting test. Its catalog family's trials in the ledger count too (for MA crossover, the 63 trend trials run so far).

In [ ]:
candidate = SleeveSpec(id="eth_ma_4h", instrument="kraken_futures:ETH/USD", interval="4h", strategy="moving_average_crossover",
                       params={"short_window": 8, "long_window": 96}, long_only=True,
                       sizing="vol_target", sizing_params={"target_annual_vol": 0.5})
card = stc.card_sleeve("config/portfolio.multi_paper.toml", candidate, ledger=RECORD)
print(card.verdict, "-", card.reason, "| baseline:", card.versus_baseline)
card.table

In [ ]:
card.variants.round(2)          # the grid: raw and Romano-Wolf adjusted p-values, against zero and against buy-and-hold

In [ ]:
card.plot();

How to read it:

- **t and the Sharpe interval** decide whether there is anything at all. Below t = 2, or an interval reaching below zero, the result is what noise looks like over this much history.
- **Deflated Sharpe** is the probability that the Sharpe beats the best of all the trials by luck. The more that was tried in the family, the higher the bar.
- **Romano-Wolf** asks the same per variant, allowing for how alike the variants are (neighbouring grid points are nearly the same strategy).
- **Overfitting (PBO)** splits history in 16 blocks, picks the best variant on every half and checks its rank on the other half. Near 0.5 means picking the best was a coin flip.
- **Timing placebo**: a long-only trend sleeve in a bull market can make money just by being long. If shifted positions do as well, the timing adds nothing.
- **Shrunk Sharpe**: the backtest Sharpe pulled toward zero by how much evidence there is. Use it, not the raw Sharpe, as the expectation for the paper review (`[review] backtest_sharpe`).

## 2. A sleeve already in a book

The same call with the sleeve's id. Every sleeve in the running books was carded on 2026-10-19 (`scripts/research/card_books.py`); the summary is below in section 4.

In [ ]:
card = stc.card_sleeve("config/portfolio.btc_live.toml", "btc_ma_1d", ledger=RECORD)
print(card.verdict, "-", card.reason)
pd.Series(card.summary()).to_frame("btc_ma_1d").T.round(2)

## 3. Any return series

For a strategy that isn't a sleeve yet (a study's output, a notebook backtest): daily net returns after costs, the baseline it must beat, every variant tried, and if possible the positions and market returns for the placebo. Here: the equity trend study H13, from the files it saves.

In [ ]:
study = sorted(Path("data/research").glob("h13_*/daily_net.csv"))[-1].parent
net = pd.read_csv(study / "daily_net.csv", index_col=0, parse_dates=True)
positions = pd.read_csv(study / "positions.csv", index_col=0, header=[0, 1], parse_dates=True)["P"]
market = pd.read_csv(study / "market_returns.csv", index_col=0, parse_dates=True)
card = stc.strategy_card(net["P"], name="H13 P", baseline=net["B1"], variants=net, sweep=net, family="H13",
                         positions=positions, asset_returns=market, periods_per_year=252, ledger=RECORD)
print(card.verdict, "-", card.reason)
card.table

## 4. Every running book at once

`python scripts/research/card_books.py config/portfolio.btc_live.toml config/portfolio.multi_paper.toml config/portfolio.small_paper.toml` cards every sleeve and basket (about a minute) and writes a summary. The run of 2026-10-19:

In [ ]:
summary = pd.read_csv("data/research/cards_running_books/summary.csv", index_col=0)
summary[~summary["verdict"].str.contains("same as")].round(2)

## 5. The gate

`--portfolio-check` lists each sleeve's latest card; `--require-strategy-cards` fails when one has no card, or "no evidence" or "short history". Run it on a new paper config before starting the book.

In [ ]:
import subprocess
result = subprocess.run([sys.executable, "main.py", "--portfolio-check", "config/portfolio.multi_paper.toml", "--require-strategy-cards"],
                        capture_output=True, text=True)
print(result.stdout[result.stdout.find("Strategy cards"):], "exit code:", result.returncode)